# Sanity Check - Step 06: ICA Artifact Removal

Überprüft:
- ICA Komponenten identifiziert
- EOG Artefakte erkannt
- Vorher/Nachher Signalqualität
- Amplituden-Reduktion durch ICA

In [ ]:
import sys
from pathlib import Path
import mne
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent / 'eeg_pipeline'))
import config

print("Setup erfolgreich")

## 1. Before & After laden

In [ ]:
subject_id = config.SUBJECTS[0]
person = "P1"

before_path = config.OUTPUT_DIR / f"sub-{subject_id}_{person}_filtered.fif"
after_path = config.OUTPUT_DIR / f"sub-{subject_id}_{person}_ica_cleaned.fif"
ica_path = config.QC_DIR / f"sub-{subject_id}_{person}_ica.fif"

if before_path.exists() and after_path.exists():
    raw_before = mne.io.read_raw_fif(str(before_path), preload=False)
    raw_after = mne.io.read_raw_fif(str(after_path), preload=False)
    print(f"✓ Both files loaded for {person}")

if ica_path.exists():
    ica = mne.preprocessing.read_ica(str(ica_path))
    print(f"✓ ICA decomposition loaded")
    print(f"  Components: {ica.n_components_}")
    print(f"  Excluded: {len(ica.exclude)}")

## 2. Metadata Überprüfung

In [ ]:
print(f"=== VORHER (Gefiltert) ===")
print(f"Kanäle: {len(raw_before.ch_names)}")
print(f"  EEG: {len(mne.pick_types(raw_before.info, eeg=True))}")
print(f"  EOG: {len(mne.pick_types(raw_before.info, eog=True))}")
print(f"Dauer: {raw_before.times[-1]:.2f}s")

print(f"\n=== NACHHER (ICA Cleaned) ===")
print(f"Kanäle: {len(raw_after.ch_names)}")
print(f"  EEG: {len(mne.pick_types(raw_after.info, eeg=True))}")
print(f"  EOG: {len(mne.pick_types(raw_after.info, eog=True))}")
print(f"Dauer: {raw_after.times[-1]:.2f}s")

print(f"\n=== VALIDIERUNGEN ===")
if len(raw_before.ch_names) == len(raw_after.ch_names):
    print(f"✓ Kanal-Anzahl erhalten")
if raw_before.n_times == raw_after.n_times:
    print(f"✓ Sample-Anzahl erhalten")

## 3. Amplituden-Vergleich

In [ ]:
eeg_picks = mne.pick_types(raw_before.info, eeg=True)
t_end = min(120, raw_before.times[-1])
t_idx_end = int(t_end * raw_before.info['sfreq'])

data_before = raw_before.get_data(picks=eeg_picks, start=0, stop=t_idx_end)
data_after = raw_after.get_data(picks=eeg_picks, start=0, stop=t_idx_end)

std_before = np.std(data_before)
std_after = np.std(data_after)
change = (std_after/std_before - 1) * 100

print(f"EEG Amplituden (erste 120s):")
print(f"  VORHER: {std_before:.6f} µV (Std)")
print(f"  NACHHER: {std_after:.6f} µV (Std)")
print(f"  Change: {change:+.1f}%")

# EOG
eog_picks = mne.pick_types(raw_before.info, eog=True)
if len(eog_picks) > 0:
    eog_data_before = raw_before.get_data(picks=eog_picks, start=0, stop=t_idx_end)
    eog_data_after = raw_after.get_data(picks=eog_picks, start=0, stop=t_idx_end)
    
    eog_std_before = np.std(eog_data_before)
    eog_std_after = np.std(eog_data_after)
    eog_reduction = (1 - eog_std_after/eog_std_before) * 100
    
    print(f"\nEOG Amplituden (Augenbewegungen):")
    print(f"  VORHER: {eog_std_before:.6f} µV (Std)")
    print(f"  NACHHER: {eog_std_after:.6f} µV (Std)")
    print(f"  Reduktion: {eog_reduction:.1f}%")